In [ ]:
# =========================
# BLOCK 1: imports + config
# =========================

import os
import json
import math
import random
from pathlib import Path
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils import parameters_to_vector, vector_to_parameters
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

@dataclass
class Config:
    data_dir: str = "/content/data"
    out_dir: str = "/content/ll_calibration"

    seed: int = 52
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    num_workers: int = 2

    # training
    epochs: int = 100
    stabilize_epochs: int = 5
    batch_size: int = 128
    eval_batch_size: int = 256
    lr: float = 0.1
    stabilize_lr: float = 1e-4
    momentum: float = 0.9
    weight_decay: float = 5e-4

    # splits
    train_fraction: float = 0.9
    analysis_subset_size: int = 1024

    # geometry
    d: int = 10
    power_iters: int = 50
    power_tol: float = 1e-5

    psi: float = 0.03
    epsilon: float = 0.02

    lipschitz_num_dirs: int = 100
    lipschitz_t_values: tuple = (1e-4, 5e-4)
    lipschitz_quantile: float = 0.95

    # monte carlo
    mc_samples: int = 500
    alpha_grid: tuple = (0.02, 0.07, 0.2, 0.5, 0.8, 1.0, 1.2, 1.3, 1.5, 1.65, 1.8, 2.2)

cfg = Config()

def ensure_dir(path):
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

OUT = ensure_dir(cfg.out_dir)
set_seed(cfg.seed)

print("device:", cfg.device)
print("out dir:", OUT)
print(torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

device: cuda
out dir: /content/ll_calibration
True
Tesla T4


In [ ]:
# =========================
# BLOCK 2: data + helpers
# =========================

def build_transforms():
    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
    ])
    eval_tf = transforms.Compose([
        transforms.ToTensor(),
    ])
    return train_tf, eval_tf

def make_cifar_resnet18(num_classes=10):
    model = models.resnet18(weights=None, num_classes=num_classes)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    return model

def trainable_params(model):
    return [p for p in model.parameters() if p.requires_grad]

def get_param_vector(model):
    return parameters_to_vector(trainable_params(model)).detach().clone()

@torch.no_grad()
def set_param_vector(model, theta):
    vector_to_parameters(theta, trainable_params(model))

def normalize(v, eps=1e-12):
    return v / (torch.linalg.vector_norm(v) + eps)

def orthogonalize(v, basis):
    if len(basis) == 0:
        return v
    B = torch.stack(basis, dim=1)
    coeffs = B.T @ v
    return v - B @ coeffs

def stratified_subset_indices(targets, candidate_indices, subset_size, seed):
    rng = np.random.default_rng(seed)
    targets_np = np.asarray(targets)
    classes = np.unique(targets_np)
    per_class = subset_size // len(classes)
    picked = []

    for c in classes:
        cls_idx = [i for i in candidate_indices if targets_np[i] == c]
        rng.shuffle(cls_idx)
        picked.extend(cls_idx[:per_class])

    leftover = subset_size - len(picked)
    if leftover > 0:
        remaining = list(set(candidate_indices) - set(picked))
        rng.shuffle(remaining)
        picked.extend(remaining[:leftover])

    return sorted(picked)

train_tf, eval_tf = build_transforms()

full_train_aug = datasets.CIFAR10(root=cfg.data_dir, train=True, download=True, transform=train_tf)
full_train_eval = datasets.CIFAR10(root=cfg.data_dir, train=True, download=True, transform=eval_tf)
test_eval = datasets.CIFAR10(root=cfg.data_dir, train=False, download=True, transform=eval_tf)

n_total = len(full_train_aug)
indices = np.arange(n_total)
rng = np.random.default_rng(cfg.seed)
rng.shuffle(indices)

n_train = int(cfg.train_fraction * n_total)
train_idx = sorted(indices[:n_train].tolist())
val_idx = sorted(indices[n_train:].tolist())

analysis_idx = stratified_subset_indices(
    targets=full_train_eval.targets,
    candidate_indices=train_idx,
    subset_size=cfg.analysis_subset_size,
    seed=cfg.seed + 1,
)

datasets_dict = {
    "train_aug": Subset(full_train_aug, train_idx),
    "train_eval": Subset(full_train_eval, train_idx),
    "val_eval": Subset(full_train_eval, val_idx),
    "analysis_eval": Subset(full_train_eval, analysis_idx),
    "test_eval": test_eval,
}

loaders = {
    "train": DataLoader(datasets_dict["train_aug"], batch_size=cfg.batch_size, shuffle=True,
                        num_workers=cfg.num_workers, pin_memory=True),
    "train_eval": DataLoader(datasets_dict["train_eval"], batch_size=cfg.eval_batch_size, shuffle=False,
                             num_workers=cfg.num_workers, pin_memory=True),
    "val": DataLoader(datasets_dict["val_eval"], batch_size=cfg.eval_batch_size, shuffle=False,
                      num_workers=cfg.num_workers, pin_memory=True),
    "analysis": DataLoader(datasets_dict["analysis_eval"], batch_size=cfg.eval_batch_size, shuffle=False,
                           num_workers=cfg.num_workers, pin_memory=True),
    "test": DataLoader(datasets_dict["test_eval"], batch_size=cfg.eval_batch_size, shuffle=False,
                       num_workers=cfg.num_workers, pin_memory=True),
}

with open(OUT / "splits.json", "w") as f:
    json.dump({
        "train_idx": train_idx,
        "val_idx": val_idx,
        "analysis_idx": analysis_idx,
    }, f)

print("train:", len(train_idx), "val:", len(val_idx), "analysis:", len(analysis_idx))

100%|██████████| 170M/170M [00:04<00:00, 42.4MB/s]


train: 45000 val: 5000 analysis: 1024


In [ ]:
# =========================
# BLOCK 3: train model
# =========================

def evaluate_loader(model, loader, device):
    model.eval()
    total_loss = 0.0
    total_acc = 0.0
    total_n = 0

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            logits = model(x)
            loss = F.cross_entropy(logits, y, reduction="sum")
            total_loss += loss.item()
            total_acc += (logits.argmax(dim=1) == y).sum().item()
            total_n += y.size(0)

    return {"loss": total_loss / total_n, "acc": total_acc / total_n}

def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    total_acc = 0.0
    total_n = 0

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        loss.backward()
        optimizer.step()

        n = y.size(0)
        total_loss += loss.item() * n
        total_acc += (logits.argmax(dim=1) == y).sum().item()
        total_n += n

    return {"loss": total_loss / total_n, "acc": total_acc / total_n}

model = make_cifar_resnet18().to(cfg.device)

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=cfg.lr,
    momentum=cfg.momentum,
    weight_decay=cfg.weight_decay,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs)

history = []
best_val_loss = float("inf")
best_path = OUT / "best_model.pt"

for epoch in range(cfg.epochs):
    train_stats = train_one_epoch(model, loaders["train"], optimizer, cfg.device)
    val_stats = evaluate_loader(model, loaders["val"], cfg.device)
    scheduler.step()

    row = {
        "epoch": epoch,
        "phase": "main",
        "lr": optimizer.param_groups[0]["lr"],
        "train_loss": train_stats["loss"],
        "train_acc": train_stats["acc"],
        "val_loss": val_stats["loss"],
        "val_acc": val_stats["acc"],
    }
    history.append(row)

    if val_stats["loss"] < best_val_loss:
        best_val_loss = val_stats["loss"]
        torch.save({
            "model_state": model.state_dict(),
            "config": asdict(cfg),
            "val_loss": best_val_loss,
        }, best_path)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(row)

ckpt = torch.load(best_path, map_location=cfg.device)
model.load_state_dict(ckpt["model_state"])

stab_opt = torch.optim.SGD(
    model.parameters(),
    lr=cfg.stabilize_lr,
    momentum=cfg.momentum,
    weight_decay=cfg.weight_decay,
)

for epoch in range(cfg.stabilize_epochs):
    train_stats = train_one_epoch(model, loaders["train"], stab_opt, cfg.device)
    val_stats = evaluate_loader(model, loaders["val"], cfg.device)
    row = {
        "epoch": epoch,
        "phase": "stabilize",
        "lr": stab_opt.param_groups[0]["lr"],
        "train_loss": train_stats["loss"],
        "train_acc": train_stats["acc"],
        "val_loss": val_stats["loss"],
        "val_acc": val_stats["acc"],
    }
    history.append(row)
    print(row)

torch.save({
    "model_state": model.state_dict(),
    "config": asdict(cfg),
}, OUT / "w_star.pt")

pd.DataFrame(history).to_csv(OUT / "training_history.csv", index=False)

print("saved:", OUT / "w_star.pt")
print("test:", evaluate_loader(model, loaders["test"], cfg.device))

{'epoch': 0, 'phase': 'main', 'lr': 0.09997532801828658, 'train_loss': 1.9781058905707465, 'train_acc': 0.2986666666666667, 'val_loss': 1.881775180053711, 'val_acc': 0.3304}
{'epoch': 4, 'phase': 'main', 'lr': 0.09938441702975688, 'train_loss': 0.8417860862414042, 'train_acc': 0.7022222222222222, 'val_loss': 0.886951107788086, 'val_acc': 0.6902}
{'epoch': 9, 'phase': 'main', 'lr': 0.09755282581475769, 'train_loss': 0.5084562556054857, 'train_acc': 0.825, 'val_loss': 0.7654852111816406, 'val_acc': 0.7432}
{'epoch': 14, 'phase': 'main', 'lr': 0.0945503262094184, 'train_loss': 0.4255976182354821, 'train_acc': 0.8537555555555556, 'val_loss': 0.5238420852661133, 'val_acc': 0.8192}
{'epoch': 19, 'phase': 'main', 'lr': 0.09045084971874741, 'train_loss': 0.3752577468236287, 'train_acc': 0.8716666666666667, 'val_loss': 0.6259094497680664, 'val_acc': 0.7992}
{'epoch': 24, 'phase': 'main', 'lr': 0.0853553390593274, 'train_loss': 0.3401165952973896, 'train_acc': 0.8824888888888889, 'val_loss': 0.4

In [ ]:
# =========================
# BLOCK 4: Hessian top eigenspace
# =========================

if "model" not in globals():
    model = make_cifar_resnet18().to(cfg.device)
    ckpt = torch.load(OUT / "w_star.pt", map_location=cfg.device)
    model.load_state_dict(ckpt["model_state"])

model.eval()
base_theta = get_param_vector(model).to(cfg.device)

def subset_loss(model, loader, device, create_graph=False):
    model.eval()
    total_loss = None
    total_n = 0

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = model(x)
        batch_loss = F.cross_entropy(logits, y, reduction="sum")
        total_loss = batch_loss if total_loss is None else total_loss + batch_loss
        total_n += y.size(0)

    return total_loss / total_n

def hvp(model, loader, vec, device):
    params = trainable_params(model)
    loss = subset_loss(model, loader, device, create_graph=True)
    grads = torch.autograd.grad(loss, params, create_graph=True, retain_graph=True)
    flat_grads = torch.cat([g.reshape(-1) for g in grads])

    dot = torch.dot(flat_grads, vec)
    hv = torch.autograd.grad(dot, params, retain_graph=False)

    flat_hv = []
    for p, g in zip(params, hv):
        if g is None:
            flat_hv.append(torch.zeros_like(p).reshape(-1))
        else:
            flat_hv.append(g.reshape(-1))
    return torch.cat(flat_hv).detach()

def top_eigenspace(model, loader, d, device, n_iters=12, tol=1e-5):
    theta0 = get_param_vector(model).to(device)
    p = theta0.numel()

    eigvals = []
    eigvecs = []

    for k in range(d):
        v = torch.randn(p, device=device)
        v = orthogonalize(v, eigvecs)
        v = normalize(v)

        prev_lambda = None
        for _ in range(n_iters):
            Hv = hvp(model, loader, v, device)
            Hv = orthogonalize(Hv, eigvecs)

            lam = torch.dot(v, Hv).item()
            Hv_norm = torch.linalg.vector_norm(Hv).item()
            if Hv_norm < 1e-12:
                raise RuntimeError(f"degenerate direction at k={k}")

            v = Hv / Hv_norm

            if prev_lambda is not None:
                rel_change = abs(lam - prev_lambda) / max(1.0, abs(lam))
                if rel_change < tol:
                    break
            prev_lambda = lam

        Hv = hvp(model, loader, v, device)
        lam = torch.dot(v, Hv).item()
        eigvals.append(lam)
        eigvecs.append(v.detach())
        print(f"eig {k+1}/{d}: lambda={lam:.6e}")

    lambdas = torch.tensor(eigvals, device=device)
    U = torch.stack(eigvecs, dim=1)
    return lambdas, U

lambdas, U = top_eigenspace(
    model=model,
    loader=loaders["analysis"],
    d=cfg.d,
    device=cfg.device,
    n_iters=cfg.power_iters,
    tol=cfg.power_tol,
)

torch.save({
    "lambdas": lambdas.cpu(),
    "U": U.cpu(),
}, OUT / "eigenspace.pt")

def subset_gradient(model, loader, device):
    params = trainable_params(model)
    loss = subset_loss(model, loader, device, create_graph=False)
    grads = torch.autograd.grad(loss, params, create_graph=False, retain_graph=False)
    flat_grad = torch.cat([g.reshape(-1) for g in grads]).detach()
    return flat_grad

print("saved:", OUT / "eigenspace.pt")
print("lambdas:", lambdas.detach().cpu().numpy())

eig 1/10: lambda=4.705269e+00
eig 2/10: lambda=3.094107e+00
eig 3/10: lambda=2.274524e+00
eig 4/10: lambda=2.055277e+00
eig 5/10: lambda=1.706987e+00
eig 6/10: lambda=1.649488e+00
eig 7/10: lambda=1.464041e+00
eig 8/10: lambda=1.369908e+00
eig 9/10: lambda=1.262929e+00
eig 10/10: lambda=1.119831e+00
saved: /content/ll_calibration/eigenspace.pt
lambdas: [4.7052693 3.0941074 2.2745245 2.0552773 1.7069873 1.6494876 1.4640415
 1.3699076 1.2629294 1.119831 ]


In [ ]:
# =========================
# BLOCK 5: estimate L_H, compute R_psi and sigma_star
# =========================

if "model" not in globals():
    model = make_cifar_resnet18().to(cfg.device)
    ckpt = torch.load(OUT / "w_star.pt", map_location=cfg.device)
    model.load_state_dict(ckpt["model_state"])
    model.eval()
    base_theta = get_param_vector(model).to(cfg.device)

if "lambdas" not in globals() or "U" not in globals():
    eig = torch.load(OUT / "eigenspace.pt", map_location=cfg.device)
    lambdas = eig["lambdas"].to(cfg.device)
    U = eig["U"].to(cfg.device)

def restricted_hessian_matrix(model, loader, U, base_theta, step_subspace, device):
    d = U.shape[1]
    theta = base_theta + U @ step_subspace
    set_param_vector(model, theta)

    cols = []
    for j in range(d):
        uj = U[:, j]
        Huj = hvp(model, loader, uj, device)
        col = U.T @ Huj
        cols.append(col)

    H_d = torch.stack(cols, dim=1)
    set_param_vector(model, base_theta)
    return H_d

def estimate_local_hessian_lipschitz(model, loader, U, base_theta, device,
                                    t_values, num_dirs, quantile=0.95):
    d = U.shape[1]
    H0 = restricted_hessian_matrix(
        model, loader, U, base_theta,
        step_subspace=torch.zeros(d, device=device),
        device=device
    )

    estimates = []

    for j in range(num_dirs):
        v = normalize(torch.randn(d, device=device))
        for t in t_values:
            Ht = restricted_hessian_matrix(
                model, loader, U, base_theta,
                step_subspace=t * v,
                device=device
            )
            diff = Ht - H0
            spec = torch.linalg.svdvals(diff)[0].item()
            estimates.append(spec / t)
        print(f"dir {j+1}/{num_dirs} done")

    arr = np.asarray(estimates)
    return {
        "strict_max": float(arr.max()),
        "robust_quantile": float(np.quantile(arr, quantile)),
        "median": float(np.median(arr)),
        "all_values": arr.tolist(),
    }

L_stats = estimate_local_hessian_lipschitz(
    model=model,
    loader=loaders["analysis"],
    U=U,
    base_theta=base_theta,
    device=cfg.device,
    t_values=cfg.lipschitz_t_values,
    num_dirs=cfg.lipschitz_num_dirs,
    quantile=cfg.lipschitz_quantile,
)

# ВАЖНО: консервативный вариант
L_hat = L_stats["strict_max"]

lambda_min = float(torch.min(lambdas).item())
R_psi = 3.0 * cfg.psi * lambda_min / L_hat
chi_quant = chi2.ppf(1.0 - cfg.epsilon, cfg.d)
sigma_star = R_psi / math.sqrt(chi_quant)

geometry_meta = {
    "lambda_min": lambda_min,
    "L_hat_strict_max": L_stats["strict_max"],
    "L_hat_robust_quantile": L_stats["robust_quantile"],
    "L_hat_median": L_stats["median"],
    "L_hat_used": L_hat,
    "R_psi": R_psi,
    "chi2_quantile": float(chi_quant),
    "sigma_star": sigma_star,
    "psi": cfg.psi,
    "epsilon": cfg.epsilon,
    "d": cfg.d,
}

with open(OUT / "geometry_meta.json", "w") as f:
    json.dump(geometry_meta, f, indent=2)

print(json.dumps(geometry_meta, indent=2))

dir 1/100 done
dir 2/100 done
dir 3/100 done
dir 4/100 done
dir 5/100 done
dir 6/100 done
dir 7/100 done
dir 8/100 done
dir 9/100 done
dir 10/100 done
dir 11/100 done
dir 12/100 done
dir 13/100 done
dir 14/100 done
dir 15/100 done
dir 16/100 done
dir 17/100 done
dir 18/100 done
dir 19/100 done
dir 20/100 done
dir 21/100 done
dir 22/100 done
dir 23/100 done
dir 24/100 done
dir 25/100 done
dir 26/100 done
dir 27/100 done
dir 28/100 done
dir 29/100 done
dir 30/100 done
dir 31/100 done
dir 32/100 done
dir 33/100 done
dir 34/100 done
dir 35/100 done
dir 36/100 done
dir 37/100 done
dir 38/100 done
dir 39/100 done
dir 40/100 done
dir 41/100 done
dir 42/100 done
dir 43/100 done
dir 44/100 done
dir 45/100 done
dir 46/100 done
dir 47/100 done
dir 48/100 done
dir 49/100 done
dir 50/100 done
dir 51/100 done
dir 52/100 done
dir 53/100 done
dir 54/100 done
dir 55/100 done
dir 56/100 done
dir 57/100 done
dir 58/100 done
dir 59/100 done
dir 60/100 done
dir 61/100 done
dir 62/100 done
dir 63/100 done
d

In [ ]:
# =========================
# BLOCK 6: Monte Carlo calibration
# =========================

if "model" not in globals():
    model = make_cifar_resnet18().to(cfg.device)
    ckpt = torch.load(OUT / "w_star.pt", map_location=cfg.device)
    model.load_state_dict(ckpt["model_state"])
    model.eval()
    base_theta = get_param_vector(model).to(cfg.device)

if "lambdas" not in globals() or "U" not in globals():
    eig = torch.load(OUT / "eigenspace.pt", map_location=cfg.device)
    lambdas = eig["lambdas"].to(cfg.device)
    U = eig["U"].to(cfg.device)

with open(OUT / "geometry_meta.json", "r") as f:
    geometry_meta = json.load(f)

R_psi = geometry_meta["R_psi"]
sigma_star = geometry_meta["sigma_star"]

base_loss = subset_loss(model, loaders["analysis"], cfg.device, create_graph=False).item()
base_grad = subset_gradient(model, loaders["analysis"], cfg.device)

rows = []

for alpha in cfg.alpha_grid:
    sigma = alpha * sigma_star
    print(f"alpha={alpha}, sigma={sigma:.6e}")

    for sample_id in range(cfg.mc_samples):
        # sampling in the d-dimensional top-curvature subspace
        z = torch.randn(cfg.d, device=cfg.device) * sigma
        h = U @ z

        z_norm = torch.linalg.vector_norm(z).item()
        h_norm = torch.linalg.vector_norm(h).item()

        theta = base_theta + h
        set_param_vector(model, theta)

        true_loss = subset_loss(model, loaders["analysis"], cfg.device, create_graph=False).item()
        delta_loss = true_loss - base_loss

        # quadratic model in the subspace
        quad_pred = 0.5 * torch.sum(lambdas * z * z).item()

        # линейный член grad^T h
        linear_term = torch.dot(base_grad, h).item()

        # полная квадратичная аппроксимация:
        # L(w*) + grad^T h + 1/2 h^T H h
        quad_approx_full = base_loss + linear_term + quad_pred

        # истинное значение L(w* + h)
        true_loss_value = true_loss

        # абсолютная ошибка по приращению
        abs_error = abs(delta_loss - (linear_term + quad_pred))

        # старая метрика для диагностики
        rel_error_old = abs_error / (abs(linear_term + quad_pred) + 1e-12)

        # устойчивая ошибка по приращению
        scale = max(abs(delta_loss), abs(linear_term + quad_pred), 1e-8)
        rel_error = abs_error / scale

        # НОВАЯ метрика:
        # |L(w)| / |L(w*) + grad^T h + 1/2 h^T H h| + delta
        quad_accuracy_ratio = abs(true_loss_value) / (abs(quad_approx_full) + 1e-8)

        # ещё полезнее иметь симметричную метрику близости
        quad_match_ratio = abs(true_loss_value - quad_approx_full) / max(
            abs(true_loss_value), abs(quad_approx_full), 1e-8
        )

        inside_ball = int(h_norm <= R_psi)
        usable_point = int(abs(quad_pred) >= 1e-6)
        valid_quad = int(rel_error <= cfg.psi)

        rows.append({
            "alpha": alpha,
            "sigma": sigma,
            "sample_id": sample_id,
            "z_norm": z_norm,
            "h_norm": h_norm,
            "inside_ball": inside_ball,
            "usable_point": usable_point,
            "true_delta_loss": delta_loss,
            "quadratic_pred": quad_pred,
            "abs_error": abs_error,
            "rel_error_old": rel_error_old,
            "rel_error": rel_error,
            "valid_quad": valid_quad,
            "linear_term": linear_term,
            "quad_approx_full": quad_approx_full,
            "true_loss_value": true_loss_value,
            "quad_accuracy_ratio": quad_accuracy_ratio,
            "quad_match_ratio": quad_match_ratio,
        })

    set_param_vector(model, base_theta)

mc_df = pd.DataFrame(rows)
mc_df.to_csv(OUT / "mc_samples.csv", index=False)

def summarize_group(g):
    inside = g[g["inside_ball"] == 1]
    usable = g[g["usable_point"] == 1]
    inside_usable = g[(g["inside_ball"] == 1) & (g["usable_point"] == 1)]

    return pd.Series({
        "sigma": g["sigma"].iloc[0],

        "p_ball": g["inside_ball"].mean(),

        "p_quad_all": g["valid_quad"].mean(),
        "p_quad_inside": inside["valid_quad"].mean() if len(inside) > 0 else np.nan,
        "p_quad_usable": usable["valid_quad"].mean() if len(usable) > 0 else np.nan,
        "p_quad_inside_usable": inside_usable["valid_quad"].mean() if len(inside_usable) > 0 else np.nan,

        "median_rel_error_all": g["rel_error"].median(),
        "q90_rel_error_all": g["rel_error"].quantile(0.9),

        "median_rel_error_inside": inside["rel_error"].median() if len(inside) > 0 else np.nan,
        "q90_rel_error_inside": inside["rel_error"].quantile(0.9) if len(inside) > 0 else np.nan,

        "median_rel_error_inside_usable": inside_usable["rel_error"].median() if len(inside_usable) > 0 else np.nan,
        "q90_rel_error_inside_usable": inside_usable["rel_error"].quantile(0.9) if len(inside_usable) > 0 else np.nan,

        "median_abs_error_all": g["abs_error"].median(),
        "q90_abs_error_all": g["abs_error"].quantile(0.9),

        "median_abs_error_inside": inside["abs_error"].median() if len(inside) > 0 else np.nan,
        "q90_abs_error_inside": inside["abs_error"].quantile(0.9) if len(inside) > 0 else np.nan,
        "median_quad_accuracy_ratio_inside": inside["quad_accuracy_ratio"].median() if len(inside) > 0 else np.nan,
        "q90_quad_accuracy_ratio_inside": inside["quad_accuracy_ratio"].quantile(0.9) if len(inside) > 0 else np.nan,

        "median_quad_match_ratio_inside": inside["quad_match_ratio"].median() if len(inside) > 0 else np.nan,
        "q90_quad_match_ratio_inside": inside["quad_match_ratio"].quantile(0.9) if len(inside) > 0 else np.nan,

        "mean_h_norm": g["h_norm"].mean(),
        "mean_z_norm": g["z_norm"].mean(),

        "n_all": len(g),
        "n_inside": len(inside),
        "n_usable": len(usable),
        "n_inside_usable": len(inside_usable),
    })

summary = mc_df.groupby("alpha").apply(summarize_group).reset_index()
summary.to_csv(OUT / "mc_summary.csv", index=False)

summary

alpha=0.02, sigma=3.791122e-07
alpha=0.07, sigma=1.326893e-06
alpha=0.2, sigma=3.791122e-06
alpha=0.5, sigma=9.477805e-06
alpha=0.8, sigma=1.516449e-05
alpha=1.0, sigma=1.895561e-05
alpha=1.2, sigma=2.274673e-05
alpha=1.3, sigma=2.464229e-05
alpha=1.5, sigma=2.843342e-05
alpha=1.65, sigma=3.127676e-05
alpha=1.8, sigma=3.412010e-05
alpha=2.2, sigma=4.170234e-05


/tmp/ipykernel_4888/3336407906.py:152: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  summary = mc_df.groupby("alpha").apply(summarize_group).reset_index()


,alpha,sigma,p_ball,p_quad_all,p_quad_inside,p_quad_usable,p_quad_inside_usable,median_rel_error_all,q90_rel_error_all,median_rel_error_inside,...,median_quad_accuracy_ratio_inside,q90_quad_accuracy_ratio_inside,median_quad_match_ratio_inside,q90_quad_match_ratio_inside,mean_h_norm,mean_z_norm,n_all,n_inside,n_usable,n_inside_usable
0,0.02,3.791122e-07,1.000,0.158,0.158000,NaN,NaN,0.101678,0.244565,0.101678,...,0.999988,0.999990,0.000001,0.000003,0.000001,0.000001,500.0,500.0,0.0,0.0
1,0.07,1.326893e-06,1.000,0.274,0.274000,NaN,NaN,0.054765,0.203618,0.054765,...,0.999988,0.999990,0.000002,0.000004,0.000004,0.000004,500.0,500.0,0.0,0.0
2,0.20,3.791122e-06,1.000,0.536,0.536000,NaN,NaN,0.026951,0.125894,0.026951,...,0.999987,0.999989,0.000002,0.000005,0.000012,0.000012,500.0,500.0,0.0,0.0
3,0.50,9.477805e-06,1.000,0.802,0.802000,NaN,NaN,0.010041,0.062253,0.010041,...,0.999987,0.999989,0.000003,0.000005,0.000030,0.000030,500.0,500.0,0.0,0.0
4,0.80,1.516449e-05,1.000,0.860,0.860000,NaN,NaN,0.007681,0.051332,0.007681,...,0.999987,0.999989,0.000003,0.000005,0.000047,0.000047,500.0,500.0,0.0,0.0
5,1.00,1.895561e-05,0.982,0.858,0.857434,NaN,NaN,0.007118,0.043362,0.007103,...,0.999987,0.999989,0.000003,0.000005,0.000058,0.000058,500.0,491.0,0.0,0.0
6,1.20,2.274673e-05,0.864,0.896,0.898148,NaN,NaN,0.004798,0.030765,0.004798,...,0.999987,0.999989,0.000003,0.000005,0.000070,0.000070,500.0,432.0,0.0,0.0
7,1.30,2.464229e-05,0.762,0.900,0.887139,NaN,NaN,0.004634,0.030378,0.005192,...,0.999987,0.999989,0.000003,0.000006,0.000076,0.000076,500.0,381.0,0.0,0.0
8,1.50,2.843342e-05,0.496,0.914,0.899194,NaN,NaN,0.003372,0.021333,0.004121,...,0.999987,0.999990,0.000003,0.000005,0.000087,0.000087,500.0,248.0,0.0,0.0
9,1.65,3.127676e-05,0.324,0.944,0.925926,NaN,NaN,0.003117,0.016091,0.004419,...,0.999987,0.999989,0.000003,0.000006,0.000098,0.000098,500.0,162.0,0.0,0.0


In [ ]:
# =========================
# BLOCK 8: resume helpers
# =========================

def load_model_from_disk():
    model = make_cifar_resnet18().to(cfg.device)
    ckpt = torch.load(OUT / "w_star.pt", map_location=cfg.device)
    model.load_state_dict(ckpt["model_state"])
    model.eval()
    return model

def load_eigenspace_from_disk():
    eig = torch.load(OUT / "eigenspace.pt", map_location=cfg.device)
    lambdas = eig["lambdas"].to(cfg.device)
    U = eig["U"].to(cfg.device)
    return lambdas, U

def load_geometry_meta():
    with open(OUT / "geometry_meta.json", "r") as f:
        return json.load(f)

print("w_star exists:", os.path.exists(OUT / "w_star.pt"))
print("eigenspace exists:", os.path.exists(OUT / "eigenspace.pt"))
print("geometry meta exists:", os.path.exists(OUT / "geometry_meta.json"))
print("mc summary exists:", os.path.exists(OUT / "mc_summary.csv"))

w_star exists: True
eigenspace exists: True
geometry meta exists: True
mc summary exists: True


In [ ]:
import shutil
from pathlib import Path

archive_base = "/content/ll_calibration_backup"
archive_path = shutil.make_archive(archive_base, "zip", root_dir=str(OUT))

print("Created archive:", archive_path)

Created archive: /content/ll_calibration_backup.zip


In [ ]:
from google.colab import files
files.download("/content/ll_calibration_backup.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import shutil
from pathlib import Path

drive_dir = Path("/content/drive/MyDrive/ll_calibration_backup")
if drive_dir.exists():
    shutil.rmtree(drive_dir)

shutil.copytree(OUT, drive_dir)
print("Copied to:", drive_dir)

Copied to: /content/drive/MyDrive/ll_calibration_backup
